# Stage 7: Evaluation & Interpretability
**Project:** KAN-Driven Phase-Spectrum Analysis for Deepfake Detection

In [ ]:
!pip install pykan kagglehub -q

In [ ]:
import numpy as np, pandas as pd, cv2, os, glob, json
import torch, torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from kan import KAN
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, accuracy_score, roc_curve, classification_report, confusion_matrix
import matplotlib.pyplot as plt, matplotlib.gridspec as gridspec, seaborn as sns
from tqdm.notebook import tqdm
import kagglehub

%matplotlib inline
plt.rcParams['figure.dpi']=120; sns.set_style('whitegrid')

INPUT_DIR = kagglehub.dataset_download('awsaf49/artifact-dataset')
OUTPUT_DIR = '/kaggle/working' if os.path.isdir('/kaggle/working') else './output'
CACHE_DIR = os.path.join(OUTPUT_DIR,'phase_cache')
MODEL_DIR = os.path.join(OUTPUT_DIR,'models')
ABL_DIR = os.path.join(MODEL_DIR,'ablations')
RPT_DIR = os.path.join(MODEL_DIR,'final_report')
os.makedirs(RPT_DIR, exist_ok=True)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

all_results={}
for fn in ['all_results.json','results_kan.json','results_b3.json']:
    p=os.path.join(MODEL_DIR,fn)
    if os.path.exists(p):
        with open(p) as f: d=json.load(f)
        if isinstance(d,dict) and 'model' in d: all_results[d['model']]=d
        else: all_results.update(d)
abl_data={}
for af in glob.glob(os.path.join(ABL_DIR,'*.csv')): abl_data[os.path.splitext(os.path.basename(af))[0]]=pd.read_csv(af)
print(f'Models:{len(all_results)} Ablations:{list(abl_data.keys())}')

In [ ]:
# Cell 2: Comparison
rows=[]
for k,r in all_results.items():
    name=r.get('model',k)
    if 'RGB' in name: dom='Spatial'
    elif 'Mag' in name: dom='Freq-Mag'
    else: dom='Freq-Phase'
    arch='KAN' if 'KAN' in name else ('ViT' if 'ViT' in name else ('ResNet' if 'Res' in name else 'MLP'))
    rows.append({'Model':name,'Arch':arch,'Domain':dom,'AUC':r.get('test_auc',0),'Accuracy':r.get('test_accuracy',0),'Params':r.get('n_parameters',0)})
cdf=pd.DataFrame(rows).sort_values('AUC',ascending=False)
print('='*70+'\nFINAL COMPARISON\n'+'='*70); print(cdf.to_string(index=False))

fig=plt.figure(figsize=(18,10))
gs=gridspec.GridSpec(2,3,figure=fig,hspace=0.35,wspace=0.3)
fig.suptitle('KAN-Phase Deepfake Detection: Final Results',fontsize=16,fontweight='bold',y=0.98)
pal={'KAN':'#e74c3c','MLP':'#3498db','ResNet':'#2ecc71','ViT':'#9b59b6'}

ax1=fig.add_subplot(gs[0,0]); colors=[pal.get(r['Arch'],'gray') for _,r in cdf.iterrows()]
ax1.barh(cdf['Model'],cdf['AUC'],color=colors);ax1.set_xlabel('AUC');ax1.set_xlim(0,1.05);ax1.set_title('Test AUC')
ax2=fig.add_subplot(gs[0,1])
for _,r in cdf.iterrows(): ax2.scatter(r['Params'],r['AUC'],s=120,c=pal.get(r['Arch'],'gray'),edgecolors='k',lw=0.5,zorder=5); ax2.annotate(r['Model'][:15],(r['Params'],r['AUC']),fontsize=7,xytext=(5,5),textcoords='offset points')
ax2.set_xlabel('Params');ax2.set_ylabel('AUC');ax2.set_xscale('log');ax2.set_title('Efficiency')
ax3=fig.add_subplot(gs[0,2]); cdf.groupby('Domain')['AUC'].mean().plot(kind='bar',ax=ax3,color=['steelblue','coral','gray'])
ax3.set_ylabel('Mean AUC');ax3.set_title('Domain');ax3.set_xticklabels(ax3.get_xticklabels(),rotation=0)
ax4=fig.add_subplot(gs[1,0])
if 'ablation_a4' in abl_data: a4=abl_data['ablation_a4'].sort_values('AUC'); ax4.barh(a4['Arch'],a4['AUC'],color='teal')
ax4.set_xlabel('AUC');ax4.set_title('A4: Arch Sweep')
ax5=fig.add_subplot(gs[1,1])
if 'ablation_a5_jpeg' in abl_data: ax5.plot(abl_data['ablation_a5_jpeg']['Quality'],abl_data['ablation_a5_jpeg']['AUC'],'o-',lw=2,label='JPEG')
if 'ablation_a5_blur' in abl_data: ax5b=ax5.twiny(); ax5b.plot(abl_data['ablation_a5_blur']['Kernel'],abl_data['ablation_a5_blur']['AUC'],'s--',lw=2,color='coral',label='Blur')
ax5.set_xlabel('JPEG Q');ax5.set_ylabel('AUC');ax5.set_ylim(0,1.05);ax5.set_title('A5: Robustness')
ax6=fig.add_subplot(gs[1,2])
if 'ablation_a6_loo' in abl_data:
    loo=abl_data['ablation_a6_loo'].sort_values('OOD_AUC')
    ax6.barh(loo['Generator'],loo['OOD_AUC'],color=['#2ecc71' if a>0.7 else '#e74c3c' for a in loo['OOD_AUC']])
    ax6.axvline(x=0.5,color='k',ls='--',alpha=0.5)
ax6.set_title('A6: OOD')
fig.savefig(os.path.join(RPT_DIR,'final_overview.png'),dpi=150,bbox_inches='tight');plt.show()
cdf.to_csv(os.path.join(RPT_DIR,'final_comparison.csv'),index=False)

In [ ]:
# Cell 3: Per-Generator AUC
cache_files=sorted(glob.glob(os.path.join(CACHE_DIR,'phase_maps_*.npy')))
pr=np.load(cache_files[-1],allow_pickle=True).item()
meta_files=glob.glob(os.path.join(INPUT_DIR,'**','metadata.csv'),recursive=True)
mdf2=pd.concat([pd.read_csv(mf).assign(generator=os.path.basename(os.path.dirname(mf))) for mf in meta_files],ignore_index=True) if meta_files else None
rg=set()
for g in pr:
    if mdf2 is not None:
        gdf=mdf2[mdf2['generator']==g]
        if len(gdf)>0 and gdf['target'].mode().iloc[0]==0: rg.add(g)
    elif 'real' in g.lower(): rg.add(g)
Xa,ya=[],[]
for g,maps in pr.items():
    for m in maps: Xa.append(m.flatten()); ya.append(0 if g in rg else 1)
Xa=np.array(Xa,dtype=np.float32); ya=np.array(ya)
sc=StandardScaler(); pc=PCA(n_components=128,random_state=42)
Xpca=pc.fit_transform(sc.fit_transform(Xa)).astype(np.float32)
Xtv,Xte,ytv,yte=train_test_split(Xpca,ya,test_size=0.2,stratify=ya,random_state=42)
Xtr,Xva,ytr,yva=train_test_split(Xtv,ytv,test_size=0.15,stratify=ytv,random_state=42)

kan=KAN(width=[128,64,32,1],grid=5,k=3,device=str(DEVICE))
cr=nn.BCEWithLogitsLoss();op=torch.optim.Adam(kan.parameters(),lr=1e-3,weight_decay=1e-4)
ld=DataLoader(TensorDataset(torch.FloatTensor(Xtr),torch.LongTensor(ytr)),batch_size=64,shuffle=True)
for _ in tqdm(range(30),desc='KAN'):
    kan.train()
    for xb,yb in ld: xb,yb=xb.to(DEVICE),yb.float().to(DEVICE);op.zero_grad();cr(kan(xb).squeeze(-1),yb).backward();op.step()

rf=[pc.transform(sc.transform(m.flatten().reshape(1,-1))).astype(np.float32)[0] for g in rg for m in pr[g][:50]]
rf=np.array(rf)
pg=[]
kan.eval()
for g in sorted(pr.keys()):
    if g in rg: continue
    ff=np.array([pc.transform(sc.transform(m.flatten().reshape(1,-1))).astype(np.float32)[0] for m in pr[g]])
    Xe=np.vstack([rf,ff]); ye=np.array([0]*len(rf)+[1]*len(ff))
    with torch.no_grad(): p=torch.sigmoid(kan(torch.FloatTensor(Xe).to(DEVICE)).squeeze(-1)).cpu().numpy()
    pg.append({'Generator':g,'AUC':roc_auc_score(ye,p),'N':len(ff)})
pgdf=pd.DataFrame(pg).sort_values('AUC',ascending=False)
print(pgdf.to_string(index=False))
if len(pgdf)>0:
    fig,ax=plt.subplots(figsize=(10,max(4,len(pgdf)*0.4)))
    ax.barh(pgdf['Generator'],pgdf['AUC'],color=['#2ecc71' if a>0.8 else '#f39c12' if a>0.6 else '#e74c3c' for a in pgdf['AUC']])
    ax.set_xlabel('AUC');ax.set_title('Per-Generator AUC');ax.invert_yaxis()
    fig.savefig(os.path.join(RPT_DIR,'per_gen_auc.png'),dpi=150);plt.tight_layout();plt.show()
pgdf.to_csv(os.path.join(RPT_DIR,'per_generator_auc.csv'),index=False)

In [ ]:
# Cell 4: Interpretability
print('='*50+'\nKAN INTERPRETABILITY\n'+'='*50)
Xg=torch.FloatTensor(Xtr[:30]).to(DEVICE).requires_grad_(True)
out=kan(Xg).sum(); out.backward()
if Xg.grad is not None:
    imp=Xg.grad.abs().mean(dim=0).cpu().numpy()
    fig,(a1,a2)=plt.subplots(1,2,figsize=(14,5)); fig.suptitle('Feature Importance',fontweight='bold')
    a1.bar(range(len(imp)),imp,color='steelblue',alpha=0.7);a1.set_xlabel('PCA Component');a1.set_ylabel('|Grad|')
    tk=20; ti=np.argsort(imp)[-tk:][::-1]
    a2.barh(range(tk),imp[ti],color='coral');a2.set_yticks(range(tk));a2.set_yticklabels([f'PC-{i}' for i in ti]);a2.invert_yaxis()
    fig.savefig(os.path.join(RPT_DIR,'feature_importance.png'),dpi=150);plt.tight_layout();plt.show()

    comps=pc.components_[:len(imp)]
    weighted=np.zeros(comps.shape[1])
    for i in range(len(imp)): weighted+=imp[i]*np.abs(comps[i])
    sz=int(np.sqrt(len(weighted))); hm=weighted.reshape(sz,sz)
    hm=(hm-hm.min())/(hm.max()-hm.min()+1e-8)

    rg_l=list(rg); fg2=[g for g in pr if g not in rg]
    samples=[]
    if rg_l: samples.append(('Real',pr[rg_l[0]][0]))
    if fg2: samples.append((f'Fake({fg2[0][:12]})',pr[fg2[0]][0]))
    if samples:
        fig,axes=plt.subplots(2,len(samples),figsize=(6*len(samples),10))
        if len(samples)==1: axes=axes.reshape(-1,1)
        fig.suptitle('Phase Attention Heatmaps',fontweight='bold')
        for j,(label,pm) in enumerate(samples):
            axes[0,j].imshow(pm,cmap='twilight',vmin=0,vmax=1);axes[0,j].set_title(f'{label} Phase');axes[0,j].axis('off')
            im=axes[1,j].imshow(hm,cmap='hot');axes[1,j].set_title(f'{label} Attention');axes[1,j].axis('off')
            fig.colorbar(im,ax=axes[1,j],fraction=0.046,pad=0.04)
        fig.savefig(os.path.join(RPT_DIR,'attention_heatmaps.png'),dpi=150);plt.tight_layout();plt.show()

try:
    kan(torch.FloatTensor(Xtr[:5]).to(DEVICE))
    kan.plot(beta=3);plt.savefig(os.path.join(RPT_DIR,'kan_network.png'),dpi=150);plt.show()
except Exception as e: print(f'pykan plot: {e}')

In [ ]:
# Cell 5: Summary
print('\n'+'='*70+'\nFINAL SUMMARY\n'+'='*70)
print(cdf[['Model','AUC','Accuracy','Params']].to_string(index=False))
if len(cdf)>0: print(f'\nBest: {cdf.iloc[0]["Model"]} (AUC={cdf.iloc[0]["AUC"]:.4f})')
print('\nKey Findings:')
print('  - Phase spectrum provides discriminative features')
print('  - KAN learns interpretable B-spline activations')
if 'ablation_a4' in abl_data: print(f'  - Best arch: {abl_data["ablation_a4"].iloc[0]["Arch"]}')
if 'ablation_a6_loo' in abl_data: print(f'  - OOD mean AUC: {abl_data["ablation_a6_loo"]["OOD_AUC"].mean():.4f}')
print('\nSaved:'); [print(f'  {f}') for f in sorted(os.listdir(RPT_DIR))]
print('\n'+'='*70+'\nEND\n'+'='*70)